In [1]:
# Cell 1
import os
import time
import json
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv

# specific imports for this notebook
from sentence_transformers import SentenceTransformer
import faiss

load_dotenv()

# Setup device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Define paths
PROCESSED_DATA_DIR = "../data/processed"
VECTORSTORE_DIR = "../vectorstore"
os.makedirs(VECTORSTORE_DIR, exist_ok=True)

INPUT_FILE = os.path.join(PROCESSED_DATA_DIR, "chunks.json")
INDEX_FILE = os.path.join(VECTORSTORE_DIR, "faiss_index.bin")
METADATA_FILE = os.path.join(VECTORSTORE_DIR, "chunk_metadata.json")

Using device: mps
PyTorch version: 2.11.0


In [2]:
# Cell 2
print(f"Loading chunks from {INPUT_FILE}...")
start_time = time.time()

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Could not find {INPUT_FILE}. Did you run Notebook 1?")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    document_chunks = json.load(f)

# Extract just the text for encoding
texts = [chunk["chunk_text"] for chunk in document_chunks]

elapsed_time = time.time() - start_time
print(f"Loaded {len(texts)} chunks in {elapsed_time:.4f} seconds.")

Loading chunks from ../data/processed/chunks.json...
Loaded 3657 chunks in 0.0147 seconds.


In [3]:
# Cell 3
print(f"Loading SentenceTransformer model to {device}...")
start_time = time.time()

# This will download the model (~90MB) on the first run
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

elapsed_time = time.time() - start_time
print(f"Model loaded in {elapsed_time:.2f} seconds.")

# Quick test to ensure MPS is handling the model correctly
test_emb = encoder.encode("Hello, this is a test.", convert_to_tensor=True)
print(f"Embedding shape: {test_emb.shape}")
print(f"Tensor device: {test_emb.device}")

Loading SentenceTransformer model to mps...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded in 34.67 seconds.
Embedding shape: torch.Size([384])
Tensor device: mps:0


In [4]:
# Cell 4
print(f"Starting to encode {len(texts)} chunks...")
start_time = time.time()

# encode() automatically batches the inputs. 
# batch_size=64 is highly optimized for M4/16GB.
# show_progress_bar=True uses tqdm under the hood.
embeddings = encoder.encode(
    texts, 
    batch_size=64, 
    show_progress_bar=True, 
    convert_to_numpy=True, # FAISS expects numpy arrays
    normalize_embeddings=True # Normalizing helps with cosine similarity / inner product
)

elapsed_time = time.time() - start_time
print(f"Encoding complete! Generated embeddings of shape {embeddings.shape} in {elapsed_time:.2f} seconds.")

Starting to encode 3657 chunks...


Batches:   0%|          | 0/58 [00:00<?, ?it/s]

Encoding complete! Generated embeddings of shape (3657, 384) in 19.56 seconds.


In [5]:
# Cell 5
print("Building FAISS index...")
start_time = time.time()

# all-MiniLM-L6-v2 outputs 384 dimensions
embedding_dim = embeddings.shape[1] 

# Because we normalized the embeddings in Cell 4, 
# Inner Product (IndexFlatIP) behaves exactly like Cosine Similarity.
index = faiss.IndexFlatIP(embedding_dim)

# Add vectors to the index
index.add(embeddings)

elapsed_time = time.time() - start_time
print(f"FAISS index built with {index.ntotal} vectors in {elapsed_time:.4f} seconds.")

# Save the FAISS index
faiss.write_index(index, INDEX_FILE)
print(f"Saved FAISS index to {INDEX_FILE}")

# Save the metadata (the chunks list) so we can map FAISS IDs back to text later
with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(document_chunks, f, indent=4, ensure_ascii=False)
print(f"Saved metadata to {METADATA_FILE}")

Building FAISS index...
FAISS index built with 3657 vectors in 0.0015 seconds.
Saved FAISS index to ../vectorstore/faiss_index.bin
Saved metadata to ../vectorstore/chunk_metadata.json


In [6]:
# Cell 6
def search_textbooks(query, top_k=3):
    print(f"\nStudent asks: '{query}'")
    start_time = time.time()
    
    # 1. Encode the student's question into a vector
    query_vector = encoder.encode([query], normalize_embeddings=True)
    
    # 2. Search the FAISS index
    # distances = similarity scores, indices = positions in our metadata list
    distances, indices = index.search(query_vector, top_k)
    
    elapsed_time = time.time() - start_time
    print(f"Retrieved top {top_k} results in {elapsed_time:.4f} seconds.\n")
    
    # 3. Print the results
    for i in range(top_k):
        idx = indices[0][i]
        score = distances[0][i]
        result = document_chunks[idx]
        
        print(f"Result {i+1} (Score: {score:.4f})")
        print(f"Class: {result['class_level']} | Subject: {result['subject']} | File: {result['filename']}")
        print(f"Text: {result['chunk_text']}\n")

# Run a test query (Change this to something relevant to your specific PDFs!)
test_query = "What is the sun and why is it hot?"
search_textbooks(test_query)

print("-" * 50)
test_query_2 = "Tell me a story about an animal."
search_textbooks(test_query_2)


Student asks: 'What is the sun and why is it hot?'
Retrieved top 3 results in 0.2817 seconds.

Result 1 (Score: 0.4917)
Class: 4 | Subject: Evs | File: deev110.pdf
Text: day? Discuss these changes with your friends and make a list. Can you guess the brightest object in the sky? It is the Sun! In fact, it is so bright that the other stars cannot be seen when it is present in the sky. The Sun gives us light and heat. Chapter 10.indd 153Chapter 10.indd 153 30-03-2025 09:53:2830-03-2025 09:53:28 Reprint 2026-27

Result 2 (Score: 0.4095)
Class: 3 | Subject: English | File: cesa110.pdf
Text: Chapter 10. Night Let us reciteLet us recite The sun that shines all day so bright, I wonder where it goes at night. It sinks behind a distant hill And all the world grows dark and still. Unit 4_English.indd 104Unit 4_English.indd 104 4/10/2024 12:14:56 PM4/10/2024 12:14:56 PM Reprint 2026-27

Result 3 (Score: 0.3747)
Class: 1 | Subject: Maths | File: aejm110.pdf
Text: 106 I study and play with my class